# Part A — Tokenizer audit

Portable executable summary of A1–A4. Detailed prose is in `Part_A_readme.md`; the decision memo is `A4_memo.md`.

In [1]:
from pathlib import Path
import json
from IPython.display import Markdown, display

cwd = Path.cwd()
ROOT = cwd if (cwd / 'results' / 'flores_eval.json').exists() else cwd / 'partA'
assert (ROOT / 'results' / 'flores_eval.json').exists(), 'Run from partA/ or repository root'
results = json.loads((ROOT / 'results' / 'flores_eval.json').read_text(encoding='utf-8'))
langs = ('eng', 'hin', 'kan', 'tam')

def table(headers, rows):
    display(Markdown('| ' + ' | '.join(headers) + ' |\n| ' + ' | '.join('---' for _ in headers) + ' |\n' + '\n'.join('| ' + ' | '.join(map(str, row)) + ' |' for row in rows)))

print('Loaded', ROOT / 'results' / 'flores_eval.json')

Loaded G:\VIT\Intern\submition\partA\results\flores_eval.json


## A1 and A2

FLORES-200 `dev`: 997 aligned sentences each for English, Hindi, Kannada and Tamil. UTF-8 text is stripped and NFC-normalized; casing and punctuation are retained.

The audit isolates lowercasing, literal-space empty words, and macro averaging. NFC is retained as deliberate canonical preprocessing with a measured small, non-zero effect.

In [2]:
bugs = results['bugs_gpt2']
table(['lang', 'v0', 'clear lower', 'clear empty', 'clear macro', 'all cleared'], [
    [lang, *(f"{bugs[lang][key]:.4f}" for key in ('v0_all_placed', 'clear_lower_only', 'clear_empty_only', 'clear_macro_only', 'all_cleared'))]
    for lang in langs
])
nfc = results['nfc']
table(['lang', 'lines changed by NFC', 'GPT-2 token delta'], [[lang, nfc[lang]['lines_changed_by_nfc'], nfc[lang]['token_delta_nfc']] for lang in langs])

| lang | v0 | clear lower | clear empty | clear macro | all cleared |
| --- | --- | --- | --- | --- | --- |
| eng | 1.2825 | 1.2367 | 1.2826 | 1.2740 | 1.2285 |
| hin | 7.8232 | 7.8225 | 7.8260 | 7.7934 | 7.7957 |
| kan | 22.1483 | 22.1467 | 22.9456 | 21.6972 | 22.6683 |
| tam | 24.7332 | 24.7314 | 24.8669 | 24.4650 | 24.6165 |

| lang | lines changed by NFC | GPT-2 token delta |
| --- | --- | --- |
| eng | 0 | 0 |
| hin | 90 | 239 |
| kan | 10 | -51 |
| tam | 2 | -6 |

## A3 — Corrected cross-language comparison

Tokens per parallel sentence is the offline routing denominator because FLORES approximately holds meaning constant. XLM-R is illustrative evidence about multilingual vocabularies, not a proxy measurement of FLM-4B.

In [3]:
for tokenizer in ('gpt2', 'xlm-roberta-base'):
    block = results['tokenizers'][tokenizer]
    eng = block['eng']['tok_per_sentence']
    display(Markdown(f'### {tokenizer}'))
    table(['lang', 'tok/word', 'tok/grapheme', 'tok/UTF-8 byte', 'tok/sentence', 'sentence ratio vs EN'], [
        [lang, f"{block[lang]['tok_per_word']:.4f}", f"{block[lang]['tok_per_grapheme']:.4f}", f"{block[lang]['tok_per_utf8_byte']:.4f}", f"{block[lang]['tok_per_sentence']:.2f}", f"{block[lang]['tok_per_sentence']/eng:.2f}×"]
        for lang in langs
    ])

### gpt2

| lang | tok/word | tok/grapheme | tok/UTF-8 byte | tok/sentence | sentence ratio vs EN |
| --- | --- | --- | --- | --- | --- |
| eng | 1.2285 | 0.2056 | 0.2055 | 25.82 | 1.00× |
| hin | 7.7957 | 2.3279 | 0.5946 | 192.41 | 7.45× |
| kan | 22.6683 | 4.0588 | 0.9786 | 350.82 | 13.59× |
| tam | 24.6165 | 4.2043 | 0.9959 | 398.36 | 15.43× |

### xlm-roberta-base

| lang | tok/word | tok/grapheme | tok/UTF-8 byte | tok/sentence | sentence ratio vs EN |
| --- | --- | --- | --- | --- | --- |
| eng | 1.3837 | 0.2316 | 0.2314 | 29.08 | 1.00× |
| hin | 1.4888 | 0.4446 | 0.1135 | 36.74 | 1.26× |
| kan | 2.5666 | 0.4595 | 0.1108 | 39.72 | 1.37× |
| tam | 2.4227 | 0.4138 | 0.0980 | 39.21 | 1.35× |

## A4 — Decision

Reject the fixed 6× Hindi budget. Measure FLM-4B's actual tokenizer on parallel and production-like prompts before routing. In production, monitor mean input-plus-output tokens per successful task by language and matched task class. See `A4_memo.md`.